# Krishi AI — Plant Disease Detection (ResNet50 + PyTorch)
### Dataset: PlantDoc (train + test splits)

**Steps:**
1. Upload `PlantDoc kaggle.zip` using the upload cell below
2. Run all cells in order
3. Download output files at the end

> **Runtime → Change Runtime Type → T4 GPU** (before running!)

In [ ]:
# ── CELL 1: Check GPU ──────────────────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change Runtime Type > T4 GPU')

In [ ]:
# ── CELL 2: Install dependencies ───────────────────────────────
!pip install -q scikit-learn seaborn onnx

In [ ]:
# ── CELL 3: Upload dataset zip ─────────────────────────────────
# Upload your 'PlantDoc kaggle.zip' file here
from google.colab import files
print('Click "Choose Files" and select your PlantDoc kaggle.zip')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

In [ ]:
# ── CELL 4: Extract dataset ────────────────────────────────────
import zipfile, os
from pathlib import Path

DATASET_DIR = Path('plant_disease_dataset')
OUTPUT_DIR  = Path('plant_disease_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Extracting {zip_name} ...')
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(DATASET_DIR)

train_dir = DATASET_DIR / 'train'
test_dir  = DATASET_DIR / 'test'

print(f'Train classes: {len(list(train_dir.iterdir()))}')
print(f'Test  classes: {len(list(test_dir.iterdir()))}')
print('Sample classes:', [d.name for d in sorted(train_dir.iterdir())[:5]])

In [ ]:
# ── CELL 5: Config & Imports ───────────────────────────────────
import json, time, csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# CONFIG
BATCH_SIZE          = 32
EPOCHS              = 30
LR                  = 1e-4
WEIGHT_DECAY        = 1e-5
EARLY_STOP_PATIENCE = 5
LR_PATIENCE         = 3
LR_FACTOR           = 0.5
DROPOUT             = 0.5
IMG_SIZE            = 224
VAL_SPLIT           = 0.15
IMAGENET_MEAN       = [0.485, 0.456, 0.406]
IMAGENET_STD        = [0.229, 0.224, 0.225]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

In [ ]:
# ── CELL 6: Transforms ─────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print('Transforms ready.')

In [ ]:
# ── CELL 7: Data Loaders ───────────────────────────────────────
from PIL import Image

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.imgs[self.subset.indices[idx]]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

# Load full train with augmentation
full_train  = datasets.ImageFolder(str(train_dir), transform=train_tf)
class_names = full_train.classes
num_classes = len(class_names)

# 85/15 train/val split
n_val   = int(VAL_SPLIT * len(full_train))
n_train = len(full_train) - n_val
train_sub, val_sub = random_split(full_train, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(42))

val_dataset  = TransformSubset(val_sub, val_tf)
test_dataset = datasets.ImageFolder(str(test_dir), transform=val_tf)

train_loader = DataLoader(train_sub,    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Classes ({num_classes}): {class_names[:5]} ...')
print(f'Train: {n_train} | Val: {n_val} | Test: {len(test_dataset)}')

In [ ]:
# ── CELL 8: Build Model (ResNet50) ─────────────────────────────
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace FC: Dropout(0.5) -> Linear(num_classes)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(DROPOUT),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params')

In [ ]:
# ── CELL 9: Training Loop ──────────────────────────────────────
MODEL_PATH = OUTPUT_DIR / 'plant_disease_model.pth'

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=LR_PATIENCE, factor=LR_FACTOR)

history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val   = float('inf')
no_improve = 0

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if is_train: optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            if is_train: loss.backward(); optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct  += preds.eq(labels).sum().item()
            total    += inputs.size(0)
    return total_loss / total, correct / total

print('='*65)
print('  TRAINING — ResNet50 on PlantDoc')
print('='*65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
    scheduler.step(vl_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    flag = ''
    if vl_loss < best_val:
        best_val = vl_loss
        no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': vl_loss, 'val_acc': vl_acc,
            'class_names': class_names,
        }, str(MODEL_PATH))
        flag = ' << BEST SAVED'
    else:
        no_improve += 1
        flag = f' (no improve {no_improve}/{EARLY_STOP_PATIENCE})'

    print(f'Ep[{epoch:02d}/{EPOCHS}] '
          f'TrLoss={tr_loss:.4f} TrAcc={tr_acc*100:.1f}% | '
          f'VlLoss={vl_loss:.4f} VlAcc={vl_acc*100:.1f}% | '
          f'{elapsed:.0f}s{flag}')

    if no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val_loss={best_val:.4f} -> model saved to {MODEL_PATH}')

In [ ]:
# ── CELL 10: Plot Training Curves ─────────────────────────────
ep = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Krishi AI - ResNet50 Training Curves', fontsize=13, fontweight='bold')

ax1.plot(ep, history['train_loss'], label='Train', color='#2563eb')
ax1.plot(ep, history['val_loss'],   label='Val',   color='#dc2626')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, [x*100 for x in history['train_acc']], label='Train', color='#16a34a')
ax2.plot(ep, [x*100 for x in history['val_acc']],   label='Val',   color='#d97706')
ax2.set_title('Accuracy (%)'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
# ── CELL 11: Load Best Model & Evaluate on Test Set ────────────
ckpt = torch.load(str(MODEL_PATH), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        _, preds = model(inputs).max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {acc*100:.2f}%')
print()
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# ── CELL 12: Confusion Matrix ──────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
n = len(class_names)
fig_sz = max(12, n // 2)
ann_sz = max(5, 9 - n // 8)

fig, ax = plt.subplots(figsize=(fig_sz, fig_sz - 1))
sns.heatmap(cm_norm, annot=(n <= 30), fmt='.2f' if n <= 30 else '',
            xticklabels=class_names, yticklabels=class_names,
            cmap='YlOrRd', ax=ax, linewidths=0.3,
            annot_kws={'size': ann_sz})
ax.set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right', fontsize=ann_sz)
plt.yticks(rotation=0,  fontsize=ann_sz)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ── CELL 13: Save Classification Report CSV ────────────────────
report = classification_report(all_labels, all_preds,
                                target_names=class_names, output_dict=True)
csv_path = OUTPUT_DIR / 'classification_report.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['class','precision','recall','f1-score','support'])
    w.writeheader()
    for cls, m in report.items():
        if isinstance(m, dict):
            w.writerow({'class': cls,
                        'precision': round(m.get('precision', 0), 4),
                        'recall':    round(m.get('recall',    0), 4),
                        'f1-score':  round(m.get('f1-score',  0), 4),
                        'support':   int(m.get('support',    0))})
print(f'CSV saved: {csv_path}')

In [ ]:
# ── CELL 14: Save label_map.json + class_names.json ────────────
with open(OUTPUT_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
with open(OUTPUT_DIR / 'label_map.json', 'w') as f:
    json.dump({str(i): n for i, n in enumerate(class_names)}, f, indent=2)
print('class_names.json + label_map.json saved.')

In [ ]:
# ── CELL 15: Export to ONNX ────────────────────────────────────
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
onnx_path = OUTPUT_DIR / 'plant_disease_model.onnx'
torch.onnx.export(
    model, dummy, str(onnx_path),
    export_params=True, opset_version=11,
    do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)
print(f'ONNX model saved: {onnx_path}')

In [ ]:
# ── CELL 16: Download All Output Files ─────────────────────────
import shutil
from google.colab import files

# Zip all outputs
shutil.make_archive('krishi_ai_outputs', 'zip', OUTPUT_DIR)
print('Zipping outputs ...')

# List files
print('\nOutput files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:<45} {f.stat().st_size/1024:>8.1f} KB')

# Download
files.download('krishi_ai_outputs.zip')
print('\nDownload started! Extract and place in your vsproject folder.')

## After Training — What to do with the output files

Extract `krishi_ai_outputs.zip` and place these files in your project:

| File | Where to put it |
|---|---|
| `plant_disease_model.pth` | `vsproject/` |
| `class_names.json` | `vsproject/` |
| `label_map.json` | `vsproject/` |
| `plant_disease_model.onnx` | `vsproject/` (optional) |
| `training_curves.png` | Keep for report |
| `confusion_matrix.png` | Keep for report |
| `classification_report.csv` | Keep for report |